In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LVXgK7nFeIQOkwIpcDuq")
project = rf.workspace("yolov7test-u13vc").project("weapon-detection-m7qso")
version = project.version(16)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to weapon-detection-16 in yolov8:: 100%|██████████| 33250/33250 [00:07<00:00, 4340.16it/s]


In [3]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.5 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano — fastest to train, best fit for a free T4 + 16.6k images

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=25,           # start modest given 16.6k images; bump later if time allows
    imgsz=640,
    batch=16,
    patience=5,          # stop early if val loss stalls, saves Colab time
    name="weapon_detect_v1"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weapon-detection-16/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=Non

In [5]:
# check instance counts per class in your training set
import yaml
with open(f"{dataset.location}/data.yaml") as f:
    print(yaml.safe_load(f))

!find {dataset.location}/train/labels -name "*.txt" | xargs cat | awk '{print $1}' | sort | uniq -c

{'names': ['gun', 'heavy-weapon', 'knife'], 'nc': 3, 'roboflow': {'license': 'CC BY 4.0', 'project': 'weapon-detection-m7qso', 'url': 'https://universe.roboflow.com/yolov7test-u13vc/weapon-detection-m7qso/dataset/16', 'version': 16, 'workspace': 'yolov7test-u13vc'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}
find: ‘{dataset.location}/train/labels’: No such file or directory


In [6]:
loc = dataset.location
print(loc)  # sanity check the actual path

/content/weapon-detection-16


In [8]:
import subprocess

labels_path = f"{loc}/train/labels"
print(labels_path)
print(os.path.exists(labels_path) if 'os' in dir() else "check next")

/content/weapon-detection-16/train/labels
check next


In [9]:
import os
from collections import Counter

labels_dir = os.path.join(loc, "train", "labels")
print("Looking in:", labels_dir)
print("Exists:", os.path.exists(labels_dir))

counts = Counter()
for fname in os.listdir(labels_dir):
    if fname.endswith(".txt"):
        with open(os.path.join(labels_dir, fname)) as f:
            for line in f:
                class_id = line.strip().split()[0]
                counts[class_id] += 1

# map back to class names using data.yaml order: gun=0, heavy-weapon=1, knife=2
names = ["gun", "heavy-weapon", "knife"]
for class_id, count in sorted(counts.items()):
    print(names[int(class_id)], ":", count)

Looking in: /content/weapon-detection-16/train/labels
Exists: True
gun : 2644
heavy-weapon : 10678
knife : 13


In [10]:
from google.colab import files
files.download("/content/runs/detect/weapon_detect_v1/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
with open("/content/runs/detect/weapon_detect_v1/results.csv") as f:
    print(f.read())

epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
1,219.675,1.39281,2.06955,1.66315,0.39737,0.26196,0.26194,0.11499,2.00588,6.73944,2.19594,0.000475635,0.000475635,0.000475635
2,430.552,1.45003,1.80983,1.71363,0.39725,0.24656,0.24389,0.1115,2.08555,7.75247,2.32966,0.00091427,0.00091427,0.00091427
3,635.738,1.43612,1.76646,1.71648,0.49188,0.28261,0.32075,0.16705,1.94288,6.66808,2.12922,0.00131518,0.00131518,0.00131518
4,839.534,1.38965,1.66198,1.6701,0.46934,0.30568,0.33804,0.17174,1.8461,5.91503,2.08603,0.00125923,0.00125923,0.00125923
5,1056.18,1.34019,1.58488,1.63472,0.48699,0.33478,0.35804,0.1992,1.78939,8.01461,1.99286,0.00120265,0.00120265,0.00120265
6,1268.62,1.29472,1.49451,1.58966,0.47519,0.31273,0.35465,0.17877,1.87651,6.58601,2.12441,0.00114606,0.00114606,0.00114606
7,1478.27,1.25968,1.44149,1.5613,0.88632,0.35874,0.42757,0.25948,1.5849